# MiniGPT — Treinamento no Google Colab

Notebook completo pra treinar o MiniGPT usando a GPU gratuita do Colab.

**Pipeline completo:**
1. Pré-treinamento (corpus sintético ou externo)
2. SFT (Supervised Fine-Tuning) — opcional
3. DPO (Direct Preference Optimization) — opcional

**Novidades:**
- BPE Tokenizer (3-5x mais eficiente que char-level)
- RoPE (Rotary Position Embedding)
- Flash Attention (PyTorch 2.0+)
- Top-p, Repetition Penalty, Typical Sampling, Beam Search
- Gradient Accumulation, Validação, Early Stopping

**Como usar:**
1. Vá em `Ambiente de execução > Alterar o tipo de ambiente de execução` e selecione **T4 GPU**
2. Execute cada célula em ordem (Shift+Enter)
3. No final, baixe os arquivos de output

In [ ]:
#@title 1. Verificar GPU
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA disponível: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
    print(f'Flash Attention: {hasattr(torch.nn.functional, "scaled_dot_product_attention")}')
else:
    print('⚠️ GPU não detectada! Vá em Ambiente de execução > Alterar tipo > T4 GPU')

In [ ]:
#@title 2. Criar todos os arquivos do projeto
%%bash
mkdir -p minigpt/data minigpt/output

cat > minigpt/config.py << 'PYEOF'
"""config.py — Central de hiperparâmetros do MiniGPT"""
from dataclasses import dataclass

@dataclass
class GPTConfig:
    d_model: int = 128
    n_heads: int = 4
    n_layers: int = 4
    context_len: int = 256
    dropout: float = 0.1
    batch_size: int = 64
    learning_rate: float = 3e-4
    weight_decay: float = 0.1
    max_epochs: int = 30
    beta1: float = 0.9
    beta2: float = 0.95
    max_grad_norm: float = 1.0
    warmup_steps: int = 100
    min_lr: float = 1e-5
    gradient_accum_steps: int = 1
    val_split: float = 0.1
    patience: int = 5
    top_p: float = 0.9
    repetition_penalty: float = 1.2
    typical_mass: float = 1.0
    tokenizer_type: str = "bpe"
    bpe_vocab_size: int = 500
    use_rope: bool = True
    beam_width: int = 1
    sft_lr: float = 1e-5
    sft_epochs: int = 5
    dpo_beta: float = 0.1
    dpo_lr: float = 1e-6
    dpo_epochs: int = 3
    vocab_size: int = 0

    @property
    def head_dim(self) -> int:
        assert self.d_model % self.n_heads == 0, (
            f"d_model ({self.d_model}) deve ser divisível por n_heads ({self.n_heads})"
        )
        return self.d_model // self.n_heads
PYEOF

echo "✓ config.py"

In [ ]:
#@title 3. Criar tokenizer.py
%%bash
cat > minigpt/tokenizer.py << 'PYEOF'
"""tokenizer.py — CharTokenizer + BPETokenizer"""
import json, re
from collections import Counter
from pathlib import Path

TOKENS_ESPECIAIS = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]

class CharTokenizer:
    def __init__(self):
        self.char_to_id = {}
        self.id_to_char = {}
        self.vocab_size = 0
    def treinar(self, texto):
        chars_unicos = sorted(set(texto))
        self.char_to_id = {}
        self.id_to_char = {}
        for i, token in enumerate(TOKENS_ESPECIAIS):
            self.char_to_id[token] = i
            self.id_to_char[i] = token
        for i, ch in enumerate(chars_unicos, start=len(TOKENS_ESPECIAIS)):
            self.char_to_id[ch] = i
            self.id_to_char[i] = ch
        self.vocab_size = len(self.char_to_id)
    def codificar(self, texto):
        return [self.char_to_id.get(ch, 1) for ch in texto]
    def decodificar(self, ids):
        return "".join(self.id_to_char.get(id_, "") for id_ in ids if self.id_to_char.get(id_, "") not in TOKENS_ESPECIAIS)
    def salvar(self, caminho):
        Path(caminho).write_text(json.dumps({"tipo": "char", "char_to_id": self.char_to_id, "id_to_char": {str(k): v for k, v in self.id_to_char.items()}, "vocab_size": self.vocab_size}, ensure_ascii=False, indent=2))
    @classmethod
    def carregar(cls, caminho):
        dados = json.loads(Path(caminho).read_text())
        tok = cls()
        tok.char_to_id = dados["char_to_id"]
        tok.id_to_char = {int(k): v for k, v in dados["id_to_char"].items()}
        tok.vocab_size = dados["vocab_size"]
        return tok
    def __repr__(self):
        return f"CharTokenizer(vocab_size={self.vocab_size})"

class BPETokenizer:
    def __init__(self):
        self.merges = []
        self.vocab = {}
        self.id_to_token = {}
        self.vocab_size = 0
    def treinar(self, texto, vocab_size=500):
        chunks = re.findall(r'\S+|\s+', texto)
        freq_chunks = Counter(chunks)
        chunk_splits = {c: list(c) for c in freq_chunks}
        todos_chars = sorted(set(c for pieces in chunk_splits.values() for c in pieces))
        base_tokens = list(TOKENS_ESPECIAIS) + todos_chars
        self.vocab = {t: i for i, t in enumerate(base_tokens)}
        self.id_to_token = {i: t for i, t in enumerate(base_tokens)}
        self.merges = []
        while len(self.vocab) < vocab_size:
            pair_freq = Counter()
            for chunk, count in freq_chunks.items():
                pieces = chunk_splits[chunk]
                for i in range(len(pieces) - 1):
                    pair_freq[(pieces[i], pieces[i+1])] += count
            if not pair_freq:
                break
            melhor_par = pair_freq.most_common(1)[0][0]
            self.merges.append(melhor_par)
            novo_token = melhor_par[0] + melhor_par[1]
            self.vocab[novo_token] = len(self.vocab)
            self.id_to_token[len(self.id_to_token)] = novo_token
            for chunk in chunk_splits:
                pieces = chunk_splits[chunk]
                new_pieces, i = [], 0
                while i < len(pieces):
                    if i < len(pieces) - 1 and pieces[i] == melhor_par[0] and pieces[i+1] == melhor_par[1]:
                        new_pieces.append(novo_token)
                        i += 2
                    else:
                        new_pieces.append(pieces[i])
                        i += 1
                chunk_splits[chunk] = new_pieces
        self.vocab_size = len(self.vocab)
    def codificar(self, texto):
        if not texto:
            return []
        chunks = re.findall(r'\S+|\s+', texto)
        ids = []
        unk_id = self.vocab.get("<UNK>", 1)
        for chunk in chunks:
            pieces = list(chunk)
            for merge in self.merges:
                new_pieces, i = [], 0
                while i < len(pieces):
                    if i < len(pieces) - 1 and pieces[i] == merge[0] and pieces[i+1] == merge[1]:
                        new_pieces.append(merge[0] + merge[1])
                        i += 2
                    else:
                        new_pieces.append(pieces[i])
                        i += 1
                pieces = new_pieces
            for piece in pieces:
                ids.append(self.vocab.get(piece, unk_id))
        return ids
    def decodificar(self, ids):
        return "".join(self.id_to_token.get(id_, "") for id_ in ids if self.id_to_token.get(id_, "") not in TOKENS_ESPECIAIS)
    def salvar(self, caminho):
        Path(caminho).write_text(json.dumps({"tipo": "bpe", "merges": [[a, b] for a, b in self.merges], "vocab": self.vocab, "id_to_token": {str(k): v for k, v in self.id_to_token.items()}, "vocab_size": self.vocab_size}, ensure_ascii=False, indent=2))
    @classmethod
    def carregar(cls, caminho):
        dados = json.loads(Path(caminho).read_text())
        tok = cls()
        tok.merges = [tuple(m) for m in dados["merges"]]
        tok.vocab = dados["vocab"]
        tok.id_to_token = {int(k): v for k, v in dados["id_to_token"].items()}
        tok.vocab_size = dados["vocab_size"]
        return tok
    def __repr__(self):
        return f"BPETokenizer(vocab_size={self.vocab_size}, merges={len(self.merges)})"

def criar_tokenizer(tipo="bpe"):
    return BPETokenizer() if tipo == "bpe" else CharTokenizer()

def carregar_tokenizer(caminho):
    dados = json.loads(Path(caminho).read_text())
    return BPETokenizer.carregar(caminho) if dados.get("tipo") == "bpe" else CharTokenizer.carregar(caminho)
PYEOF

echo "✓ tokenizer.py"

In [ ]:
#@title 4. Criar model.py (com RoPE + Flash Attention)
%%bash
cat > minigpt/model.py << 'PYEOF'
"""model.py — GPT com RoPE + Flash Attention"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F
from config import GPTConfig

def _rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat([-x2, x1], dim=-1)

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, max_seq_len=2048):
        super().__init__()
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2, dtype=torch.float32) / dim))
        self.register_buffer("inv_freq", inv_freq)
        self._build_cache(max_seq_len)
    def _build_cache(self, seq_len):
        t = torch.arange(seq_len, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat([freqs, freqs], dim=-1)
        self.register_buffer("cos_cached", emb.cos())
        self.register_buffer("sin_cached", emb.sin())
    def forward(self, x, offset=0):
        seq_len = x.shape[-2]
        cos = self.cos_cached[offset : offset + seq_len]
        sin = self.sin_cached[offset : offset + seq_len]
        while cos.dim() < x.dim():
            cos = cos.unsqueeze(0)
            sin = sin.unsqueeze(0)
        return x * cos + _rotate_half(x) * sin

class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.gain = nn.Parameter(torch.ones(d_model))
        self.bias = nn.Parameter(torch.zeros(d_model))
    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        return self.gain * (x - mean) / torch.sqrt(var + self.eps) + self.bias

class MultiHeadSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.d_model % config.n_heads == 0
        self.n_heads = config.n_heads
        self.head_dim = config.head_dim
        self.d_model = config.d_model
        self.use_rope = config.use_rope
        self.W_qkv = nn.Linear(config.d_model, 3 * config.d_model)
        self.W_out = nn.Linear(config.d_model, config.d_model)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        if config.use_rope:
            self.rotary_emb = RotaryEmbedding(config.head_dim, config.context_len)
        else:
            self.rotary_emb = None
        self._use_flash = hasattr(F, "scaled_dot_product_attention")

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.W_qkv(x)
        q, k, v = qkv.chunk(3, dim=-1)
        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        if self.use_rope and self.rotary_emb is not None:
            q = self.rotary_emb(q)
            k = self.rotary_emb(k)
        if self._use_flash:
            out = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=self.attn_dropout.p if self.training else 0.0)
        else:
            attn_scores = (q @ k.transpose(-2, -1)) / math.sqrt(self.head_dim)
            mascara_causal = torch.triu(torch.ones(T, T, device=x.device, dtype=torch.bool), diagonal=1)
            attn_scores = attn_scores.masked_fill(mascara_causal, float("-inf"))
            attn_probs = F.softmax(attn_scores, dim=-1)
            attn_probs = self.attn_dropout(attn_probs)
            out = attn_probs @ v
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        out = self.W_out(out)
        return self.resid_dropout(out)

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(config.d_model, 4 * config.d_model), nn.GELU(), nn.Linear(4 * config.d_model, config.d_model), nn.Dropout(config.dropout))
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.d_model)
        self.attn = MultiHeadSelfAttention(config)
        self.ln2 = LayerNorm(config.d_model)
        self.ffn = FeedForward(config)
    def forward(self, x):
        return x + self.ffn(self.ln2(x + self.attn(self.ln1(x))))

class GPTModel(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.token_embedding = nn.Embedding(config.vocab_size, config.d_model)
        if config.use_rope:
            self.position_embedding = None
        else:
            self.position_embedding = nn.Embedding(config.context_len, config.d_model)
        self.drop = nn.Dropout(config.dropout)
        self.blocks = nn.Sequential(*[TransformerBlock(config) for _ in range(config.n_layers)])
        self.ln_f = LayerNorm(config.d_model)
        self.lm_head = nn.Linear(config.d_model, config.vocab_size, bias=False)
        self.lm_head.weight = self.token_embedding.weight
        self._init_weights()

    def _init_weights(self):
        n_layers = self.config.n_layers
        for nome, modulo in self.named_modules():
            if isinstance(modulo, nn.Linear):
                nn.init.normal_(modulo.weight, mean=0.0, std=0.02)
                if modulo.bias is not None:
                    nn.init.zeros_(modulo.bias)
            elif isinstance(modulo, nn.Embedding):
                nn.init.normal_(modulo.weight, mean=0.0, std=0.02)
            if nome.endswith(("ffn.net.0", "W_qkv", "W_out")):
                nn.init.normal_(modulo.weight, mean=0.0, std=0.02 / math.sqrt(2 * n_layers))

    def forward(self, idx, targets=None):
        B, T = idx.shape
        tok_emb = self.token_embedding(idx)
        if self.position_embedding is not None:
            pos = torch.arange(0, T, device=idx.device).unsqueeze(0)
            x = self.drop(tok_emb + self.position_embedding(pos))
        else:
            x = self.drop(tok_emb)
        x = self.blocks(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-100)
        return logits, loss

    def contar_parametros(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

    def resumo(self):
        n = self.contar_parametros()
        c = self.config
        lines = [
            f"════════════════ MiniGPT — Resumo do Modelo ════════════════",
            f"  Parâmetros:     {n:,}",
            f"  Vocabulário:    {c.vocab_size}",
            f"  d_model:        {c.d_model}",
            f"  n_heads:        {c.n_heads}",
            f"  head_dim:       {c.head_dim}",
            f"  n_layers:       {c.n_layers}",
            f"  context_len:    {c.context_len}",
            f"  dropout:        {c.dropout}",
            f"  FFN dim:        {4 * c.d_model}",
            f"  RoPE:           {'Sim' if c.use_rope else 'Não'}",
            f"  Flash Attn:     {'Disponível' if hasattr(F, 'scaled_dot_product_attention') else 'Não disponível'}",
            f"════════════════════════════════════════════════════════════",
        ]
        return "\n".join(lines)
PYEOF

echo "✓ model.py"

In [ ]:
#@title 5. Criar train.py, generate.py, sft.py, dpo.py, data/corpus.py
%%bash

cat > minigpt/train.py << 'PYEOF'
"""train.py — Loop de treinamento com grad accumulation, validação, early stopping"""
import json, math, time
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from config import GPTConfig
from model import GPTModel
from tokenizer import CharTokenizer, BPETokenizer, criar_tokenizer

class TextDataset(Dataset):
    def __init__(self, tokens, context_len):
        self.tokens = tokens
        self.context_len = context_len
    def __len__(self):
        return max(0, len(self.tokens) - self.context_len)
    def __getitem__(self, idx):
        chunk = self.tokens[idx : idx + self.context_len + 1]
        return torch.tensor(chunk[:-1], dtype=torch.long), torch.tensor(chunk[1:], dtype=torch.long)

def get_lr(it, config):
    if it < config.warmup_steps:
        return config.learning_rate * (it + 1) / config.warmup_steps
    progress = min((it - config.warmup_steps) / max(1, 10000 - config.warmup_steps), 1.0)
    coeff = 0.5 * (1.0 + math.cos(math.pi * progress))
    return config.min_lr + coeff * (config.learning_rate - config.min_lr)

@torch.no_grad()
def avaliar(modelo, dataloader, device):
    modelo.eval()
    total_loss = 0.0
    n = 0
    for x, y in dataloader:
        x, y = x.to(device), y.to(device)
        _, loss = modelo(x, y)
        if loss is not None:
            total_loss += loss.item()
            n += 1
    modelo.train()
    return total_loss / max(n, 1)

def treinar(config, texto, saida_dir="output", device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
    print(f"Dispositivo: {device}")
    saida_path = Path(saida_dir)
    saida_path.mkdir(exist_ok=True)
    print("Construindo tokenizer...")
    tokenizer = criar_tokenizer(config.tokenizer_type)
    if isinstance(tokenizer, BPETokenizer):
        tokenizer.treinar(texto, vocab_size=config.bpe_vocab_size)
    else:
        tokenizer.treinar(texto)
    print(f"  {tokenizer}")
    config.vocab_size = tokenizer.vocab_size
    tokens = tokenizer.codificar(texto)
    print(f"  Corpus: {len(texto):,} caracteres → {len(tokens):,} tokens")
    val_size = int(len(tokens) * config.val_split)
    train_tokens = tokens[:-val_size] if val_size > 0 else tokens
    val_tokens = tokens[-val_size:] if val_size > 0 else None
    train_dataset = TextDataset(train_tokens, config.context_len)
    train_loader = DataLoader(train_dataset, batch_size=config.batch_size, shuffle=True, num_workers=0, pin_memory=device!="cpu")
    val_loader = None
    if val_tokens and len(val_tokens) > config.context_len:
        val_dataset = TextDataset(val_tokens, config.context_len)
        val_loader = DataLoader(val_dataset, batch_size=config.batch_size, shuffle=False, num_workers=0, pin_memory=device!="cpu")
    print(f"  Batch size: {config.batch_size} × {config.gradient_accum_steps} = {config.batch_size * config.gradient_accum_steps} efetivo")
    print(f"  Batches/época: {len(train_loader)} | Validação: {'Sim' if val_loader else 'Não'}")
    modelo = GPTModel(config).to(device)
    print(modelo.resumo())
    param_decay = [p for n, p in modelo.named_parameters() if p.dim() >= 2]
    param_nodecay = [p for n, p in modelo.named_parameters() if p.dim() < 2]
    otimizador = torch.optim.AdamW([{"params": param_decay, "weight_decay": config.weight_decay}, {"params": param_nodecay, "weight_decay": 0.0}], lr=config.learning_rate, betas=(config.beta1, config.beta2))
    modelo.train()
    step_global = 0
    melhor_val_loss = float("inf")
    melhor_train_loss = float("inf")
    epochs_sem_melhora = 0
    media_loss = 0.0
    epoca = 0
    total_tokens = 0
    tempo_inicio = time.time()
    log_epocas = []
    print(f"\nTreinando por {config.max_epochs} épocas...")
    print("=" * 70)
    for epoca in range(1, config.max_epochs + 1):
        modelo.train()
        epoca_loss = 0.0
        n_batches = 0
        t0 = time.time()
        otimizador.zero_grad(set_to_none=True)
        accum = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            logits, loss = modelo(x, y)
            (loss / config.gradient_accum_steps).backward()
            accum += 1
            epoca_loss += loss.item()
            n_batches += 1
            total_tokens += x.shape[0] * x.shape[1]
            if accum % config.gradient_accum_steps == 0:
                lr = get_lr(step_global, config)
                for pg in otimizador.param_groups:
                    pg["lr"] = lr
                torch.nn.utils.clip_grad_norm_(modelo.parameters(), config.max_grad_norm)
                otimizador.step()
                otimizador.zero_grad(set_to_none=True)
                step_global += 1
        media_loss = epoca_loss / max(n_batches, 1)
        elapsed = time.time() - t0
        ppl = math.exp(min(media_loss, 20))
        lr_atual = get_lr(step_global, config)
        val_str = ""
        val_loss = None
        if val_loader:
            val_loss = avaliar(modelo, val_loader, device)
            val_str = f" │ Val Loss: {val_loss:.4f}"
        print(f"Época {epoca:3d}/{config.max_epochs} │ Loss: {media_loss:.4f} │ PPL: {ppl:.2f} │ LR: {lr_atual:.2e} │ Tempo: {elapsed:.1f}s{val_str}")
        log_epocas.append({"epoca": epoca, "loss": round(media_loss, 6), "ppl": round(ppl, 2), "lr": lr_atual, "tempo_seg": round(elapsed, 1), **({"val_loss": round(val_loss, 6)} if val_loss is not None else {})})
        loss_monitor = val_loss if val_loss is not None else media_loss
        loss_ref = melhor_val_loss if val_loss is not None else melhor_train_loss
        if loss_monitor < loss_ref:
            if val_loss is not None:
                melhor_val_loss = val_loss
            else:
                melhor_train_loss = media_loss
            epochs_sem_melhora = 0
            torch.save({"modelo": modelo.state_dict(), "config": config, "epoch": epoca, "loss": media_loss, **({"val_loss": val_loss} if val_loss is not None else {}), "total_tokens": total_tokens, "tempo_total_seg": time.time() - tempo_inicio}, saida_path / "melhor_modelo.pt")
            tokenizer.salvar(str(saida_path / "tokenizer.json"))
        else:
            epochs_sem_melhora += 1
        if config.patience > 0 and epochs_sem_melhora >= config.patience:
            print(f"\nEarly stopping! Sem melhora há {config.patience} épocas.")
            break
    tempo_total = time.time() - tempo_inicio
    melhor = melhor_val_loss if val_loader else melhor_train_loss
    print("=" * 70)
    print(f"Concluído! Melhor loss: {melhor:.4f} | Tempo: {tempo_total:.1f}s ({tempo_total/60:.1f} min) | {total_tokens/max(tempo_total,1):,.0f} tokens/s")
    return modelo, tokenizer
PYEOF

cat > minigpt/generate.py << 'PYEOF'
"""generate.py — Geração com top-p, repetition penalty, typical sampling, beam search"""
import torch
import torch.nn.functional as F
from model import GPTModel
from tokenizer import CharTokenizer, BPETokenizer

def apply_top_k(logits, top_k):
    if top_k is None or top_k <= 0:
        return logits
    kth_vals = torch.topk(logits, min(top_k, logits.size(-1)))[0][:, -1:]
    return logits.masked_fill(logits < kth_vals, float("-inf"))

def apply_top_p(logits, top_p):
    if top_p >= 1.0:
        return logits
    sorted_logits, sorted_indices = torch.sort(logits, descending=True)
    cumulative_probs = torch.cumsum(F.softmax(sorted_logits, dim=-1), dim=-1)
    sorted_indices_to_remove = cumulative_probs > top_p
    sorted_indices_to_remove[..., 1:] = sorted_indices_to_remove[..., :-1].clone()
    sorted_indices_to_remove[..., 0] = False
    indices_to_remove = torch.zeros_like(logits, dtype=torch.bool)
    indices_to_remove.scatter_(-1, sorted_indices, sorted_indices_to_remove)
    return logits.masked_fill(indices_to_remove, float("-inf"))

def apply_repetition_penalty(logits, tokens_gerados, penalty):
    if penalty == 1.0 or not tokens_gerados:
        return logits
    for token_id in set(tokens_gerados):
        if token_id < logits.size(-1):
            if logits[0, token_id] > 0:
                logits[0, token_id] /= penalty
            else:
                logits[0, token_id] *= penalty
    return logits

@torch.no_grad()
def gerar(modelo, tokenizer, prompt, max_tokens=200, temperature=0.8, top_k=40, top_p=0.9, repetition_penalty=1.2, typical_mass=1.0, device="cpu"):
    modelo.eval()
    tokens = tokenizer.codificar(prompt)
    idx = torch.tensor([tokens], dtype=torch.long, device=device)
    eos_id = tokenizer.vocab.get("<EOS>", -1) if isinstance(tokenizer, BPETokenizer) else tokenizer.char_to_id.get("<EOS>", -1)
    tokens_gerados = list(tokens)
    for _ in range(max_tokens):
        idx_cond = idx[:, -modelo.config.context_len :]
        logits, _ = modelo(idx_cond)
        logits = logits[:, -1, :]
        logits = logits / max(temperature, 1e-8)
        logits = apply_repetition_penalty(logits, tokens_gerados, repetition_penalty)
        logits = apply_top_k(logits, top_k)
        logits = apply_top_p(logits, top_p)
        probs = F.softmax(logits, dim=-1)
        proximo_token = torch.multinomial(probs, num_samples=1)
        if proximo_token.item() == eos_id:
            break
        tokens_gerados.append(proximo_token.item())
        idx = torch.cat([idx, proximo_token], dim=1)
    return tokenizer.decodificar(idx[0].tolist())

def gerar_variacoes(modelo, tokenizer, prompt, n_variacoes=3, max_tokens=200, temperature=0.8, top_k=40, top_p=0.9, repetition_penalty=1.2, device="cpu"):
    return [gerar(modelo, tokenizer, prompt, max_tokens, temperature, top_k, top_p, repetition_penalty, device=device) for _ in range(n_variacoes)]

@torch.no_grad()
def gerar_beam_search(modelo, tokenizer, prompt, max_tokens=100, beam_width=5, temperature=1.0, length_penalty=0.6, device="cpu"):
    modelo.eval()
    tokens = tokenizer.codificar(prompt)
    eos_id = tokenizer.vocab.get("<EOS>", -1) if isinstance(tokenizer, BPETokenizer) else tokenizer.char_to_id.get("<EOS>", -1)
    beams = [(0.0, tokens)]
    for _ in range(max_tokens):
        all_candidates = []
        for score, seq in beams:
            idx = torch.tensor([seq[-modelo.config.context_len:]], dtype=torch.long, device=device)
            logits, _ = modelo(idx)
            log_probs = F.log_softmax(logits[:, -1, :] / max(temperature, 1e-8), dim=-1)
            top_probs, top_indices = torch.topk(log_probs[0], beam_width)
            for i in range(beam_width):
                all_candidates.append((score + top_probs[i].item(), seq + [top_indices[i].item()]))
        all_candidates.sort(key=lambda x: x[0] / (len(x[1]) ** length_penalty), reverse=True)
        beams = all_candidates[:beam_width]
        if all(seq[-1] == eos_id for _, seq in beams):
            break
    best_score, best_seq = max(beams, key=lambda x: x[0] / (len(x[1]) ** length_penalty))
    return tokenizer.decodificar(best_seq)
PYEOF

cat > minigpt/sft.py << 'PYEOF'
"""sft.py — Supervised Fine-Tuning"""
import time
from pathlib import Path
import torch
from torch.utils.data import Dataset, DataLoader
from config import GPTConfig
from model import GPTModel
from tokenizer import CharTokenizer, BPETokenizer

SEPARATOR = " Resposta: "
IGNORE_INDEX = -100

class SFTDataset(Dataset):
    def __init__(self, pares, tokenizer, context_len):
        self.samples = []
        self.context_len = context_len
        for instrucao, resposta in pares:
            texto_completo = f"Instrução: {instrucao}{SEPARATOR}{resposta}"
            tokens_completo = tokenizer.codificar(texto_completo)
            texto_inst = f"Instrução: {instrucao}{SEPARATOR}"
            tokens_inst = tokenizer.codificar(texto_inst)
            inst_len = len(tokens_inst)
            if len(tokens_completo) > context_len:
                continue
            input_ids = tokens_completo
            targets = tokens_completo[1:] + [0]
            targets = [-100] * min(inst_len, len(targets)) + targets[inst_len:]
            targets = targets[:len(input_ids)]
            if len(input_ids) > 0 and len(targets) > 0:
                self.samples.append((input_ids[:context_len], targets[:context_len]))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        return torch.tensor(self.samples[idx][0], dtype=torch.long), torch.tensor(self.samples[idx][1], dtype=torch.long)

def treinar_sft(modelo, tokenizer, config, dados_sft, saida_dir="output", device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
    print(f"\n{'='*60}\nSFT — Supervised Fine-Tuning\nDispositivo: {device}\nDados: {len(dados_sft)} pares\nLR: {config.sft_lr}\nÉpocas: {config.sft_epochs}\n{'='*60}\n")
    saida_path = Path(saida_dir)
    dataset = SFTDataset(dados_sft, tokenizer, config.context_len)
    if len(dataset) == 0:
        print("ERRO: Nenhuma amostra SFT válida. Aumente context_len.")
        return modelo
    dataloader = DataLoader(dataset, batch_size=min(config.batch_size, len(dataset)), shuffle=True, num_workers=0, pin_memory=device!="cpu")
    modelo = modelo.to(device)
    optimizer = torch.optim.AdamW(modelo.parameters(), lr=config.sft_lr, weight_decay=config.weight_decay)
    modelo.train()
    melhor_loss = float("inf")
    for epoca in range(1, config.sft_epochs + 1):
        total_loss = 0.0
        n = 0
        t0 = time.time()
        for x, y in dataloader:
            x, y = x.to(device), y.to(device)
            logits, loss = modelo(x, y)
            if loss is None: continue
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), config.max_grad_norm)
            optimizer.step()
            total_loss += loss.item()
            n += 1
        media = total_loss / max(n, 1)
        print(f"SFT Época {epoca:3d}/{config.sft_epochs} │ Loss: {media:.4f} │ Tempo: {time.time()-t0:.1f}s")
        if media < melhor_loss:
            melhor_loss = media
            torch.save({"modelo": modelo.state_dict(), "config": config, "epoch": epoca, "loss": media, "tipo": "sft"}, saida_path / "melhor_modelo_sft.pt")
    modelo.eval()
    print(f"\nSFT concluído! Melhor loss: {melhor_loss:.4f}")
    return modelo
PYEOF

cat > minigpt/dpo.py << 'PYEOF'
"""dpo.py — Direct Preference Optimization"""
import copy, time
from pathlib import Path
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from config import GPTConfig
from model import GPTModel
from tokenizer import CharTokenizer, BPETokenizer

class DPODataset(Dataset):
    def __init__(self, pares, tokenizer, context_len):
        self.samples = []
        self.context_len = context_len
        for prompt, chosen, rejected in pares:
            chosen_ids = tokenizer.codificar(prompt + chosen)
            rejected_ids = tokenizer.codificar(prompt + rejected)
            if len(chosen_ids) > context_len or len(rejected_ids) > context_len:
                continue
            chosen_targets = chosen_ids[1:] + [0]
            rejected_targets = rejected_ids[1:] + [0]
            self.samples.append({"chosen_ids": chosen_ids, "chosen_targets": chosen_targets, "rejected_ids": rejected_ids, "rejected_targets": rejected_targets, "prompt_len": len(tokenizer.codificar(prompt))})
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        s = self.samples[idx]
        mx = max(len(s["chosen_ids"]), len(s["rejected_ids"]), 1)
        def pad(seq, target_len, v=0): return s[seq][:target_len] + [v] * max(0, target_len - len(s[seq][:target_len]))
        return {"chosen_ids": torch.tensor(pad("chosen_ids", mx), dtype=torch.long), "chosen_targets": torch.tensor(pad("chosen_targets", mx, -100), dtype=torch.long), "rejected_ids": torch.tensor(pad("rejected_ids", mx), dtype=torch.long), "rejected_targets": torch.tensor(pad("rejected_targets", mx, -100), dtype=torch.long), "prompt_len": s["prompt_len"]}

def _log_probs(modelo, input_ids, targets):
    logits, _ = modelo(input_ids, targets)
    log_probs = F.log_softmax(logits, dim=-1)
    token_lp = log_probs.gather(2, targets.unsqueeze(-1)).squeeze(-1)
    mask = targets != -100
    return (token_lp * mask).sum(dim=-1)

def treinar_dpo(modelo, tokenizer, config, dados_dpo, saida_dir="output", device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu")
    print(f"\n{'='*60}\nDPO — Direct Preference Optimization\nDispositivo: {device}\nDados: {len(dados_dpo)} pares\nbeta: {config.dpo_beta}\n{'='*60}\n")
    saida_path = Path(saida_dir)
    modelo_ref = copy.deepcopy(modelo)
    modelo_ref.eval()
    for p in modelo_ref.parameters(): p.requires_grad = False
    modelo = modelo.to(device)
    modelo_ref = modelo_ref.to(device)
    dataset = DPODataset(dados_dpo, tokenizer, config.context_len)
    if len(dataset) == 0:
        print("ERRO: Nenhuma amostra DPO válida. Aumente context_len.")
        return modelo
    dataloader = DataLoader(dataset, batch_size=min(config.batch_size, len(dataset)), shuffle=True, num_workers=0)
    optimizer = torch.optim.AdamW([p for p in modelo.parameters() if p.requires_grad], lr=config.dpo_lr)
    modelo.train()
    beta = config.dpo_beta
    for epoca in range(1, config.dpo_epochs + 1):
        total_loss = 0.0
        n = 0
        t0 = time.time()
        for batch in dataloader:
            ci, ct, ri, rt = batch["chosen_ids"].to(device), batch["chosen_targets"].to(device), batch["rejected_ids"].to(device), batch["rejected_targets"].to(device)
            lc = _log_probs(modelo, ci, ct)
            lr = _log_probs(modelo, ri, rt)
            with torch.no_grad():
                lrc = _log_probs(modelo_ref, ci, ct)
                lrr = _log_probs(modelo_ref, ri, rt)
            loss = -F.logsigmoid(beta * ((lc - lrc) - (lr - lrr))).mean()
            optimizer.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(modelo.parameters(), config.max_grad_norm)
            optimizer.step()
            total_loss += loss.item()
            n += 1
        media = total_loss / max(n, 1)
        print(f"DPO Época {epoca:3d}/{config.dpo_epochs} │ Loss: {media:.4f} │ Tempo: {time.time()-t0:.1f}s")
    torch.save({"modelo": modelo.state_dict(), "config": config, "epoch": config.dpo_epochs, "loss": media, "tipo": "dpo"}, saida_path / "melhor_modelo_dpo.pt")
    modelo.eval()
    print(f"\nDPO concluído! Loss final: {media:.4f}")
    return modelo
PYEOF

touch minigpt/data/__init__.py

cat > minigpt/data/corpus.py << 'PYEOF'
"""data/corpus.py — Corpus sintético + dados SFT/DPO"""
import random
from pathlib import Path

def gerar_corpus():
    sujeitos = ["o gato", "a gata", "o cachorro", "a cachorra", "o pássaro", "o menino", "a menina", "o professor", "a professora", "o artista", "a cozinheira", "o cientista", "a médica", "o engenheiro", "a escritora", "o músico", "a rainha", "o rei", "a bailarina", "o pescador"]
    verbos = ["comia", "bebia", "dormia", "corria", "pulava", "brincava", "estudava", "trabalhava", "cantava", "dançava", "falava", "ouvia", "via", "pensava", "sonhava", "escrevia", "lia", "pintava", "cozinhava", "navegava"]
    objetos = ["na praça", "no parque", "na casa", "na escola", "no jardim", "na rua", "na praia", "no rio", "na montanha", "na cidade", "na floresta", "no campo", "na biblioteca", "no teatro", "na cozinha"]
    complementos = ["com alegria", "com cuidado", "com calma", "com atenção", "com amor", "com paciência", "com vontade", "sem pressa", "com determinação", "com entusiasmo", "com gratidão", "com coragem"]
    frases = []
    for s in sujeitos:
        for v in verbos:
            for o in objetos[:5]:
                frases.append(f"{s.capitalize()} {v} {o}.")
    for s in sujeitos[:5]:
        for v in verbos[:5]:
            for o in objetos[:3]:
                for c in complementos[:4]:
                    frases.append(f"{s.capitalize()} {v} {o} {c}.")
    for s in sujeitos[:5]:
        for v in verbos[:5]:
            frases.append(f"{s.capitalize()} não {v}.")
            frases.append(f"Será que {s} {v}?")
    historias = ["Era uma vez um gato que morava numa casa muito bonita. O gato gostava de dormir no jardim todas as tardes. Um dia, o gato encontrou um pássaro na árvore. O pássaro cantava uma melodia linda. O gato e o pássaro ficaram amigos para sempre.", "A menina estudava todas as noites na biblioteca. Ela lia muitos livros sobre ciência e arte. A menina queria ser uma grande cientista. A professora ajudava a menina com as dúvidas. A menina trabalhava com dedicação e alegria.", "O cachorro corria no parque todas as manhãs. Ele gostava de brincar com a bola. O cachorro era muito feliz. A dona do cachorro cuidava dele com muito amor. Juntos, eles caminhavam pela cidade.", "Na cidade grande, as pessoas corriam para todo lado. O trânsito era intenso e barulhento. Mas no parque, tudo era calma e tranquilo. As crianças brincavam na praça. Os velhos sentavam no banco e observavam os pássaros.", "O professor explicava a lição com paciência. Os estudantes ouviam com atenção. A aula era sobre a natureza e os animais. A menina fez uma pergunta. O professor respondeu com alegria. A turma aprendeu muito naquele dia.", "A cozinheira preparava um jantar delicioso. Ela cozinhava com muito cuidado. O jantar tinha arroz, feijão e salada. A família comeu com alegria.", "O artista pintava um quadro muito bonito. Ele usava cores vivas e fortes. O quadro mostrava uma paisagem da montanha.", "A cientista trabalhava no laboratório. Ela estudava as estrelas e os planetas. A cientista descobriu uma nova estrela.", "No jardim da casa, as flores cresciam com beleza. O sol brilhava e a chuva caía com cuidado. As borboletas voavam entre as flores.", "O menino e a menina brincavam na praia. Eles construíam castelos de areia. O mar era azul e calmo. As crianças estavam felizes."]
    frases.extend(historias * 5)
    frases_soltas = ["O sol brilha no céu azul.", "A chuva cai na cidade.", "O vento sopra na montanha.", "A lua ilumina a noite.", "As estrelas brilham no escuro.", "O rio corre para o mar.", "A floresta é verde e bonita.", "O fogo esquenta a casa.", "A água é importante para a vida.", "A natureza é maravilhosa.", "O dia amanheceu bonito.", "O pôr do sol era lindo.", "A noite estava estrelada.", "A manhã chegou com alegria.", "A tarde foi tranquila.", "O aprendizado nunca termina.", "A leitura abre portas.", "O conhecimento transforma vidas.", "A amizade é um tesouro.", "O amor é a maior força.", "A vida é bela e curta.", "O tempo passa depressa.", "A esperança nunca morre.", "O mundo é cheio de surpresas.", "A música alegra o coração."]
    frases.extend(frases_soltas * 15)
    dialogos = ["— Bom dia! — disse a menina com um sorriso. — Bom dia! — respondeu o menino alegremente.", "— O que você está lendo? — perguntou o professor. — Estou lendo um livro sobre ciência! — respondeu a estudante.", "— Vamos brincar no parque? — sugeriu o menino. — Sim, vamos! — disseram as crianças.", "— A comida está deliciosa! — elogiou o visitante. — Obrigada! — disse a cozinheira satisfeita.", "— Qual é o seu sonho? — perguntou a amiga. — Eu quero ser cientista! — respondeu a menina."]
    frases.extend(dialogos * 10)
    random.seed(42)
    random.shuffle(frases)
    return " ".join(frases)

def gerar_dados_sft():
    return [("O que é o sol?", "O sol é uma estrela que ilumina e aquece a Terra."), ("O que os gatos gostam de fazer?", "Os gatos gostam de dormir, brincar e receber carinho."), ("Como é a praia?", "A praia é um lugar bonito com areia e mar."), ("O que é a chuva?", "A chuva é água que cai das nuvens. É importante para as plantas e a natureza."), ("Quem é o professor?", "O professor é uma pessoa que ensina e ajuda os estudantes."), ("O que é a amizade?", "A amizade é um sentimento de afeto e confiança entre pessoas."), ("Como é a floresta?", "A floresta é um lugar cheio de árvores, plantas e animais."), ("O que é a música?", "A música é uma forma de arte que usa sons organizados de maneira harmoniosa."), ("O que você faz na escola?", "Na escola, nós estudamos, aprendemos coisas novas e fazemos amigos."), ("O que é a leitura?", "A leitura é o ato de interpretar textos escritos. Ela nos leva a mundos imaginários."), ("Como é a cidade?", "A cidade é um lugar com muitas pessoas, prédios, ruas e carros."), ("O que é a ciência?", "A ciência é o estudo do mundo natural. Os cientistas fazem experiências para entender como as coisas funcionam."), ("Por que o céu é azul?", "O céu parece azul porque a luz do sol se espalha ao passar pela atmosfera."), ("O que são as estrelas?", "As estrelas são grandes esferas de gás quente que brilham no céu."), ("Como é o inverno?", "O inverno é a estação mais fria do ano. Os dias são mais curtos e às vezes neva."), ("O que é um jardim?", "O jardim é um lugar com flores, plantas e árvores cuidadas com carinho."), ("O que é a coragem?", "A coragem é a força de enfrentar o medo e as dificuldades."), ("Como funcionam os sonhos?", "Os sonhos são imagens e histórias que o cérebro cria enquanto dormimos."), ("O que é a felicidade?", "A felicidade é um sentimento de alegria. Cada pessoa encontra a felicidade de um jeito diferente."), ("O que é a sabedoria?", "A sabedoria é o conhecimento combinado com a experiência de vida.")]

def gerar_dados_dpo():
    return [("O que é o sol?", "O sol é uma estrela que ilumina e aquece a Terra.", "O sol é uma planeta grande."), ("Como é a praia?", "A praia é um lugar bonito com areia e mar.", "A praia é um prédio grande."), ("O que os gatos gostam?", "Os gatos gostam de dormir e brincar.", "Os gatos gostam de pilotar aviões."), ("O que é a chuva?", "A chuva é água que cai das nuvens.", "A chuva é fogo que cai do céu."), ("Quem é o professor?", "O professor é uma pessoa que ensina os estudantes.", "O professor é um animal que vive na floresta."), ("O que é a amizade?", "A amizade é um sentimento de carinho e confiança.", "A amizade é uma planta que cresce no jardim."), ("O que é a ciência?", "A ciência é o estudo do mundo natural.", "A ciência é um tipo de comida."), ("Como é o inverno?", "O inverno é a estação mais fria do ano.", "O inverno é a estação mais quente do ano."), ("O que são estrelas?", "As estrelas são esferas de gás que brilham no céu.", "As estrelas são pedras no chão."), ("O que é a música?", "A música é uma arte que organiza sons de forma harmoniosa.", "A música é um esporte radical.")]

def salvar_corpus(caminho="data/corpus.txt"):
    texto = gerar_corpus()
    Path(caminho).write_text(texto, encoding="utf-8")
    return texto

def carregar_corpus(caminho="data/corpus.txt"):
    path = Path(caminho)
    if path.exists():
        return path.read_text(encoding="utf-8")
    return salvar_corpus(caminho)

def carregar_corpus_externo(caminho):
    path = Path(caminho)
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {caminho}")
    return path.read_text(encoding="utf-8")
PYEOF

echo "✓ train.py, generate.py, sft.py, dpo.py, data/corpus.py"

In [ ]:
#@title 6. Pré-treinamento { display-mode: "form" }

#@markdown ## Configuração
D_MODEL = 128        #@param {type:"integer"}
N_HEADS = 4          #@param {type:"integer"}
N_LAYERS = 4         #@param {type:"integer"}
CONTEXT_LEN = 256    #@param {type:"integer"}
BATCH_SIZE = 64        #@param {type:"integer"}
MAX_EPOCHS = 30        #@param {type:"integer"}
LEARNING_RATE = 3e-4 #@param {type:"number"}
GRADIENT_ACCUM = 1    #@param {type:"integer"}
TOKENIZER = "bpe"     #@param ["bpe", "char"]
USE_ROPE = True       #@param {type:"boolean"}

import sys
sys.path.insert(0, 'minigpt')
import torch

from config import GPTConfig
from train import treinar
from tokenizer import criar_tokenizer
from data.corpus import gerar_corpus

device = 'cuda' if torch.cuda.is_available() else 'cpu'

config = GPTConfig(
    d_model=D_MODEL, n_heads=N_HEADS, n_layers=N_LAYERS,
    context_len=CONTEXT_LEN, batch_size=BATCH_SIZE,
    max_epochs=MAX_EPOCHS, learning_rate=LEARNING_RATE,
    gradient_accum_steps=GRADIENT_ACCUM,
    tokenizer_type=TOKENIZER, use_rope=USE_ROPE,
)

texto = gerar_corpus()
print(f'Corpus: {len(texto):,} caracteres')

modelo, tokenizer = treinar(config, texto, saida_dir='minigpt/output', device=device)

In [ ]:
#@title 7. SFT — Supervised Fine-Tuning (opcional)
SFT_EPOCHS = 5        #@param {type:"integer"}
SFT_LR = 1e-5         #@param {type:"number"}

from sft import treinar_sft
from data.corpus import gerar_dados_sft
from tokenizer import carregar_tokenizer

# Carregar melhor modelo
checkpoint = torch.load('minigpt/output/melhor_modelo.pt', map_location=device, weights_only=False)
config_sft = checkpoint['config']
if not hasattr(config_sft, 'use_rope'): config_sft.use_rope = True
if not hasattr(config_sft, 'tokenizer_type'): config_sft.tokenizer_type = 'bpe'
if not hasattr(config_sft, 'sft_lr'): config_sft.sft_lr = 1e-5
if not hasattr(config_sft, 'sft_epochs'): config_sft.sft_epochs = 5
if 'position_embedding.weight' in checkpoint['modelo']: config_sft.use_rope = False
tokenizer = carregar_tokenizer('minigpt/output/tokenizer.json')
modelo = GPTModel(config_sft).to(device)
modelo.load_state_dict(checkpoint['modelo'], strict=False)

config_sft.sft_epochs = SFT_EPOCHS
config_sft.sft_lr = SFT_LR
dados = gerar_dados_sft()
print(f'SFT data: {len(dados)} pares')

modelo = treinar_sft(modelo, tokenizer, config_sft, dados, saida_dir='minigpt/output', device=device)

In [ ]:
#@title 8. DPO — Direct Preference Optimization (opcional)
DPO_EPOCHS = 3        #@param {type:"integer"}
DPO_BETA = 0.1        #@param {type:"number"}

from dpo import treinar_dpo
from data.corpus import gerar_dados_dpo
import os

# Carregar modelo (preferir SFT)
modelo_path = 'minigpt/output/melhor_modelo_sft.pt'
if not os.path.exists(modelo_path):
    modelo_path = 'minigpt/output/melhor_modelo.pt'
checkpoint = torch.load(modelo_path, map_location=device, weights_only=False)
config_dpo = checkpoint['config']
if not hasattr(config_dpo, 'use_rope'): config_dpo.use_rope = True
if not hasattr(config_dpo, 'dpo_beta'): config_dpo.dpo_beta = 0.1
if not hasattr(config_dpo, 'dpo_lr'): config_dpo.dpo_lr = 1e-6
if not hasattr(config_dpo, 'dpo_epochs'): config_dpo.dpo_epochs = 3
if 'position_embedding.weight' in checkpoint['modelo']: config_dpo.use_rope = False
tokenizer = carregar_tokenizer('minigpt/output/tokenizer.json')
modelo = GPTModel(config_dpo).to(device)
modelo.load_state_dict(checkpoint['modelo'], strict=False)

config_dpo.dpo_epochs = DPO_EPOCHS
config_dpo.dpo_beta = DPO_BETA
dados = gerar_dados_dpo()
print(f'DPO data: {len(dados)} pares')

modelo = treinar_dpo(modelo, tokenizer, config_dpo, dados, saida_dir='minigpt/output', device=device)

In [ ]:
#@title 9. Testar o modelo
modelo.eval()
modelo_cpu = modelo.to('cpu')

from generate import gerar, gerar_variacoes, gerar_beam_search

print("=== Geração com Top-p + Repetition Penalty ===")
for prompt in ["O gato", "A menina", "Na cidade", "Era uma vez", "Instrução: O que é o sol? Resposta:"]:
    resultado = gerar(modelo_cpu, tokenizer, prompt, max_tokens=120,
                      temperature=0.8, top_k=40, top_p=0.9,
                      repetition_penalty=1.2, device='cpu')
    print(f'Prompt: "{prompt}"')
    print(f'Saída:  {resultado}\n')

print("=== Beam Search ===")
for prompt in ["O gato", "A menina"]:
    resultado = gerar_beam_search(modelo_cpu, tokenizer, prompt,
                                   max_tokens=60, beam_width=5, device='cpu')
    print(f'Prompt: "{prompt}"')
    print(f'Beam:   {resultado}\n')

In [ ]:
#@title 10. Baixar resultados
from google.colab import files
import os

output_dir = 'minigpt/output'
for f in os.listdir(output_dir):
    fpath = os.path.join(output_dir, f)
    size_mb = os.path.getsize(fpath) / (1024*1024)
    print(f'{f}: {size_mb:.1f} MB')

print('\nBaixando arquivos...')
for f in os.listdir(output_dir):
    fpath = os.path.join(output_dir, f)
    files.download(fpath)